# Week 6: Fine-Tuning an LLM for Transaction Risk Scoring using an opensource LLM

---

## Details

An exercise to fine-tune `meta-llama/Llama-3.2-1B-Instruct` on a real-world AML (Anti-Money Laundering) dataset to build a transaction risk scoring assistant.


---
Colab Link: https://colab.research.google.com/drive/1hfFrQs-Qgu4jNdZb2FKKXgouEDfz-s7S

---
## Part 1: Setup

### 1.1 — Install Dependencies

In [ ]:
!pip install  -q --upgrade  bitsandbytes trl peft accelerate transformers


### 1.2 — Configuration

In [ ]:
import json
import re
import random
from enum import Enum
from typing import List
from pydantic import BaseModel, Field, field_validator
from huggingface_hub import login
from datasets import load_dataset
import pandas as pd
from google.colab import userdata

MODEL_NAME       = "meta-llama/Llama-3.2-1B-Instruct"
MAX_SEQ_LENGTH   = 1024
LORA_RANK        = 32
LORA_ALPHA       = 32
LORA_DROPOUT     = 0.05
BATCH_SIZE       = 4
GRAD_ACCUM       = 2
LEARNING_RATE    = 2e-4
NUM_EPOCHS       = 3
WARMUP_RATIO     = 0.03
SAVE_DIR         = "./transaction_risk_lora"
DATASET_SAMPLE_N = 100
RANDOM_SEED      = 42

random.seed(RANDOM_SEED)
print("✅ Config ready")

### 1.3 — Define Pydantic Output Schema

In [ ]:
class RiskLevel(str, Enum):
    LOW    = "low_risk"
    MEDIUM = "medium_risk"
    HIGH   = "high_risk"
    CRITICAL = "critical_risk"


class RecommendedAction(str, Enum):
    ALLOW  = "allow"
    OTP    = "ask_otp"
    OTP_FLAGGED = "ask_otp_and_flag_for_review"
    BLOCK  = "block"
    REVIEW = "flag_for_review"


class TransactionRiskScore(BaseModel):
    fraud_probability_value: float = Field(
        ge=0.0, le=1.0,
        description="Probability of fraud, between 0.0 and 1.0"
    )

    fraud_probability_analysis: RiskLevel = Field(
        description="Risk tier: low_risk / medium_risk / high_risk / critical_risk"
    )
    action: RecommendedAction = Field(
        description="Recommended action to take on this transaction"
    )
    reasoning: str = Field(
        min_length=20,
        description="Plain-language explanation of why this score was assigned"
    )

    risk_factors: List[str] = Field(
        default_factory=list,
        description="Bullet-point list of key risk signals identified"
    )

    @field_validator("fraud_probability_value")
    @classmethod
    def round_probability(cls, v: float) -> float:
        return round(v, 4)

    def summary(self) -> str:
        return (
            f"[{self.fraud_probability_analysis.value.upper()}] "
            f"p={self.fraud_probability_value} → {self.action.value}"
        )


class TransactionInput(BaseModel):
    """Validates raw transaction features before they are sent to the model."""
    current_transaction_amount: float
    current_transaction_time: str
    current_transaction_time_gap_from_previous_transaction_time_in_minutes: int
    account_age_in_days: int
    average_transaction_amount_in_the_analysis_in_the_past_3_months_including_current_month: float
    usual_smallest_transaction_time_gap_when_busy_analysis_in_the_past_3_months_including_current_month_in_minutes: int
    usual_time_range_of_account_transactions_in_the_analysis_in_the_past_3_months_including_current_month: str


# Quick sanity check — this should pass without errors
test_obj = TransactionRiskScore(
    fraud_probability_value    = 0.72,
    fraud_probability_analysis = "high_risk",
    action                     = "ask_otp_and_flag_for_review",
    reasoning                  = "Amount is 10x the average and transaction occurred at 2am.",
    risk_factors               = ["Unusual hour", "Amount spike"]
)
print("✅ Pydantic schema works:", test_obj.summary())

### 1.4 — Login to HuggingFace

In [ ]:
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)
print("✅ HuggingFace login successful")

---
## Part 2: Data

### 2.1 — Load the Dataset



In [ ]:
raw_dataset = load_dataset(
    "peacemakerzvidzanayi/aml_fraud_detection_and_reasoning_50k_syntetic",
    split="train"
)
df = raw_dataset.to_pandas()

print(f"Dataset loaded: {len(raw_dataset):,} records")
print(f"Columns: {raw_dataset.column_names}")
print(f"\nRisk tier distribution:")
print(df["fraud_probability_analysis"].value_counts().to_string())
print("\nSample record:")
print(json.dumps(df.iloc[0].to_dict(), indent=2, default=str))

### 2.2 — Stratified Sampling

We take 100 records from the full dataset using **stratified sampling** to ensure the model sees a balanced mix of risk tiers during training.

In [ ]:
from datasets import Dataset

# Since the finetuned model classification did not do too well (around 30% accuracy) we may need to increase this to at least 1000 for better training.
STRATA = {
    "low_risk"   : 30,
    "medium_risk": 30,
    "high_risk"  : 25,
    "critical_risk": 15,
}

sampled_frames = []
for tier, n in STRATA.items():
    tier_df  = df[df["fraud_probability_analysis"] == tier]
    n_actual = min(n, len(tier_df))
    sampled_frames.append(tier_df.sample(n=n_actual, random_state=RANDOM_SEED))
    print(f"  Sampled {n_actual} × '{tier}'")

sampled_df = (
    pd.concat(sampled_frames)
    .sample(frac=1, random_state=RANDOM_SEED)
    .reset_index(drop=True)
)
print(f"\nFinal sample: {len(sampled_df)} records")
assert len(sampled_df) == 100, "Sample must be exactly 100 records"

### 2.3 — Format Data as Chat Instructions

We convert each record into a three-turn conversation: a **system** prompt describing the task, a **user** message with the transaction features, and an **assistant** message containing the expected JSON output.


In [ ]:
SYSTEM_PROMPT = """You are an expert AML (Anti-Money Laundering) transaction risk analyst.
Given transaction features, you output a structured JSON risk assessment.
Your JSON must match this exact schema:
{
  "fraud_probability_value": <float 0.0-1.0>,
  "fraud_probability_analysis": <"low_risk" | "medium_risk" | "high_risk">,
  "action": <"allow" | "ask_otp" | "block" | "flag_for_review">,
  "reasoning": <string explaining the score>,
  "risk_factors": [<list of key risk signals>]
}
Output ONLY the JSON object. No preamble, no markdown, no explanation outside the JSON."""


def record_to_user_prompt(row: dict) -> str:
    """Formats the transaction fields into a structured user message."""
    return f"""Analyse this transaction and return a risk assessment JSON:

Transaction Amount   : {row['current_transaction_amount']}
Transaction Time     : {row['current_transaction_time']}
Time Gap (mins)      : {row['current_transaction_time_gap_from_previous_transaction_time_in_minutes']}
Account Age (days)   : {row['account_age_in_days']}
Avg Amount (3m)      : {row['average_transaction_amount_in_the_analysis_in_the_past_3_months_including_current_month']:.2f}
Usual Smallest Gap   : {row['usual_smallest_transaction_time_gap_when_busy_analysis_in_the_past_3_months_including_current_month_in_minutes']} mins
Usual Time Window    : {row['usual_time_range_of_account_transactions_in_the_analysis_in_the_past_3_months_including_current_month']}"""


def record_to_assistant_response(row: dict) -> str:
    """Builds the expected assistant JSON output from the ground-truth labels."""
    reasoning    = str(row.get("reasoning", ""))
    raw_factors  = [f.strip() for f in re.split(r'\s+and\s+', reasoning) if len(f.strip()) > 10]
    risk_factors = raw_factors[:4] if raw_factors else ["See reasoning for details"]
    output = {
        "fraud_probability_value"   : round(float(row["fraud_probability_value"]), 4),
        "fraud_probability_analysis": row["fraud_probability_analysis"],
        "action"                    : row["action"],
        "reasoning"                 : reasoning,
        "risk_factors"              : risk_factors,
    }
    return json.dumps(output, indent=2)


def format_as_chat(row: dict) -> dict:
    return {
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": record_to_user_prompt(row)},
            {"role": "assistant", "content": record_to_assistant_response(row)},
        ]
    }


formatted   = [format_as_chat(r) for r in sampled_df.to_dict(orient="records")]
hf_dataset  = Dataset.from_list(formatted)
split       = hf_dataset.train_test_split(test_size=0.1, seed=RANDOM_SEED)
train_dataset = split["train"]
eval_dataset  = split["test"]

print(f"Train: {len(train_dataset)} examples | Eval: {len(eval_dataset)} examples")
print("\nFirst training example — system turn:")
print(train_dataset[0]["messages"][0]["content"])
print("\nFirst training example — user turn:")
print(train_dataset[0]["messages"][1]["content"])

---
## Part 3: Model Setup

### 3.1 — Load the Base Model in 4-bit

We use `BitsAndBytesConfig` to load the model in 4-bit quantization to reduces VRAM from ~8 GB to ~2 GB, making training possible on a free Colab T4.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_compute_dtype    = torch.bfloat16,
    bnb_4bit_use_double_quant = True,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token            = hf_token,
    padding_side     = "right",
    model_max_length = MAX_SEQ_LENGTH,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config = bnb_config,
    device_map          = "auto",
    token               = hf_token,
    torch_dtype         = torch.float16,
)
model.config.use_cache      = False
model.config.pretraining_tp = 1

if torch.cuda.is_available():
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    used  = torch.cuda.memory_allocated(0) / 1e9
    print(f"✅ Model loaded | VRAM: {used:.2f} GB used / {total:.1f} GB total")
else:
    print("⚠️  No GPU — switch to T4 runtime before continuing")

### 3.2 — Attach LoRA Adapters
Attach LORA adapters using `peft` librarry

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    task_type    = TaskType.CAUSAL_LM,
    r            = LORA_RANK,
    lora_alpha   = LORA_ALPHA,
    lora_dropout = LORA_DROPOUT,
    bias         = "none",
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters    : {total:,}")
print(f"Trainable (LoRA)    : {trainable:,}  ({100*trainable/total:.2f}%)")
print(f"Frozen (base model) : {total-trainable:,}  ({100*(1-trainable/total):.2f}%)")

---
## Part 4: Training

### 4.1 — Tokenize the Dataset

Convert conversation into token IDs using the model's tokenizer.


In [ ]:
def tokenize_example(example):
    text = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )

    tokenized = tokenizer(
        text,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    )

    tokenized["labels"] = tokenized["input_ids"].copy()

    return tokenized


train_tokenized = train_dataset.map(
    tokenize_example,
    remove_columns = train_dataset.column_names,
    desc           = "Tokenizing train",
)
eval_tokenized = eval_dataset.map(
    tokenize_example,
    remove_columns = eval_dataset.column_names,
    desc           = "Tokenizing eval",
)

print(f"✅ Tokenized | Train: {len(train_tokenized)} | Eval: {len(eval_tokenized)}")
print(f"   Token sequence length: {len(train_tokenized[0]['input_ids'])}")

### 4.2 — Configure and Run Training

We use `TrainingArguments` from `transformers` and `SFTTrainer` from `trl`.


In [ ]:
import os
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from trl import SFTTrainer
from google.colab import drive


drive.mount('/content/drive')
SAVE_DIR = "/content/drive/MyDrive/transaction_risk_lora"
os.makedirs(SAVE_DIR, exist_ok=True)


training_args = TrainingArguments(
    output_dir                  = SAVE_DIR,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    num_train_epochs            = NUM_EPOCHS,
    warmup_steps                = 5,
    learning_rate               = LEARNING_RATE,
    fp16                        = False,
    bf16                        = True,
    logging_steps               = 5,
    eval_strategy               = "steps",   
    eval_steps                  = 50,
    save_strategy               = "steps",  
    save_steps                  = 50,
    save_total_limit            = 2,         
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    optim                       = "paged_adamw_8bit",
    weight_decay                = 0.01,
    lr_scheduler_type           = "cosine",
    seed                        = RANDOM_SEED,
    report_to                   = "none",
    remove_unused_columns       = False,
)


collator = DataCollatorForSeq2Seq(
    tokenizer,
    pad_to_multiple_of = 8,
    return_tensors     = "pt",
    padding            = True,
)

trainer = SFTTrainer(
    model         = model,
    data_collator = collator,
    train_dataset = train_tokenized,
    eval_dataset  = eval_tokenized,
    args          = training_args,
)

last_checkpoint = None
if os.path.isdir(SAVE_DIR):
    checkpoints = [
        os.path.join(SAVE_DIR, d) for d in os.listdir(SAVE_DIR)
        if d.startswith("checkpoint-")
    ]
    if checkpoints:
        last_checkpoint = max(checkpoints, key=os.path.getmtime)
        print(f"▶️  Resuming from: {last_checkpoint}")
    else:
        print("▶️  No checkpoint found — starting from scratch")
        
print("Starting training...")
trainer_stats = trainer.train(resume_from_checkpoint=last_checkpoint)

print(f"\n✅ Done | Steps: {trainer_stats.global_step} | Loss: {trainer_stats.training_loss:.4f} | Time: {trainer_stats.metrics.get('train_runtime', 0):.0f}s")

### 4.3 — Save the Model

In [ ]:
import os

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

adapter_size = sum(
    os.path.getsize(os.path.join(SAVE_DIR, f))
    for f in os.listdir(SAVE_DIR)
    if os.path.isfile(os.path.join(SAVE_DIR, f))
) / 1e6

print(f"✅ Saved locally to '{SAVE_DIR}' ({adapter_size:.1f} MB)")
print(f"   Files: {[f for f in os.listdir(SAVE_DIR) if not f.startswith('checkpoint')]}")

HF_USERNAME = "johnmboga"
HF_REPO_NAME = f"{HF_USERNAME}/transaction-risk-lora"

model.push_to_hub(
    HF_REPO_NAME,
    token      = hf_token,
    private    = False,  
    commit_message = "Add LoRA adapter fine-tuned on AML fraud detection dataset",
)
tokenizer.push_to_hub(
    HF_REPO_NAME,
    token      = hf_token,
    private    = True,
)

print(f"\n✅ Pushed to HuggingFace: https://huggingface.co/{HF_REPO_NAME}")
print(f"   Adapter only — base model ({MODEL_NAME}) is referenced, not duplicated")

---
## Part 5: Inference & Evaluation

### 5.1 — Reload and Run Inference




In [ ]:
from peft import PeftModel
from pydantic import ValidationError

HF_USERNAME = "johnmboga"
HF_REPO_NAME = f"{HF_USERNAME}/transaction-risk-lora"


base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config,
    device_map="auto", token=hf_token, torch_dtype=torch.bfloat16,
)
ft_tokenizer = AutoTokenizer.from_pretrained(HF_REPO_NAME, token=hf_token)
ft_model     = PeftModel.from_pretrained(base, HF_REPO_NAME, token=hf_token)
ft_model.eval()
print("✅ Fine-tuned model loaded from HuggingFace")


def run_inference(transaction: dict, mdl=None, tok=None, max_new_tokens: int = 400) -> str:
    mdl = mdl or ft_model
    tok = tok or ft_tokenizer
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": record_to_user_prompt(transaction)},
    ]

    prompt = tok.apply_chat_template(
        messages,
        add_generation_prompt = True,
        tokenize              = False,
    )

    input_ids = tok(
        prompt,
        return_tensors = "pt",
        truncation     = True,
        max_length     = MAX_SEQ_LENGTH,
    ).input_ids.to(mdl.device)

    with torch.no_grad():
        outputs = mdl.generate(
            input_ids,
            max_new_tokens = max_new_tokens,
            temperature    = 0.1,
            do_sample      = True,
            pad_token_id   = tok.eos_token_id,
        )
    new_tokens = outputs[0][input_ids.shape[-1]:]
    return tok.decode(new_tokens, skip_special_tokens=True).strip()


def parse_risk_response(raw_output: str) -> TransactionRiskScore:
    clean      = re.sub(r'```(?:json)?', '', raw_output).strip()
    json_match = re.search(r'\{.*\}', clean, re.DOTALL)
    if not json_match:
        raise ValueError(f"No JSON object found in model output:\n{raw_output[:300]}")
    data = json.loads(json_match.group())
    return TransactionRiskScore(**data)


def score_transaction(transaction: dict, mdl=None, tok=None):
    raw = run_inference(transaction, mdl=mdl, tok=tok)
    try:
        return parse_risk_response(raw), raw
    except (json.JSONDecodeError, ValidationError, ValueError) as e:
        print(f"⚠️  Parse error: {e}")
        return None, raw


test_record = sampled_df.iloc[5].to_dict()
score, raw  = score_transaction(test_record)
if score:
    print(f"\nTest passed ✅ → {score.summary()}")
else:
    print(f"\nParse failed ❌ — check your parse_risk_response implementation")
    print(f"Raw output: {raw[:300]}")

### 5.2 — Before vs After Comparison

Run the same transaction through the **base model** and the **fine-tuned model** and compare their outputs side-by-side.

In [ ]:
print("Loading base model for comparison...")
base_model_raw = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config,
    device_map="auto", token=hf_token, torch_dtype=torch.float16,
)
base_tok_raw = AutoTokenizer.from_pretrained(MODEL_NAME, token=hf_token, padding_side="right")
if base_tok_raw.pad_token is None:
    base_tok_raw.pad_token = base_tok_raw.eos_token
base_model_raw.eval()

test_tx = sampled_df.iloc[0].to_dict()
print(f"\nTransaction: amount={test_tx['current_transaction_amount']} | expected={test_tx['fraud_probability_analysis']} → {test_tx['action']}")

print("\n" + "─" * 30 + " BASE MODEL " + "─" * 30)
base_raw = run_inference(test_tx, mdl=base_model_raw, tok=base_tok_raw)
print(base_raw[:600])
try:
    base_score = parse_risk_response(base_raw)
    print(f"\nParsed ✅ → {base_score.summary()}")
except Exception as e:
    print(f"\nPydantic validation failed ❌: {e}")

print("\n" + "─" * 28 + " FINE-TUNED MODEL " + "─" * 28)
ft_score, ft_raw = score_transaction(test_tx)
print(ft_raw[:600])
if ft_score:
    print(f"\nParsed ✅ → {ft_score.summary()}")
    print(f"Correct tier  : {'✅' if ft_score.fraud_probability_analysis.value == test_tx['fraud_probability_analysis'] else '❌'}")
    print(f"Correct action: {'✅' if ft_score.action.value == test_tx['action'] else '❌'}")

### 5.3 — Batch Evaluation



In [ ]:
eval_records   = sampled_df.tail(10).to_dict(orient="records")
results        = []
parse_failures = 0

for i, record in enumerate(eval_records):
    score, _ = score_transaction(record)
    if score is None:
        parse_failures += 1
        results.append({"tier_match": False, "action_match": False})
    else:
        results.append({
            "tier_match"  : score.fraud_probability_analysis.value == record["fraud_probability_analysis"],
            "action_match": score.action.value == record["action"],
        })
    print(f"  [{i+1:02d}] Expected: {record['fraud_probability_analysis']:<13} "
          f"Tier: {'✅' if results[-1]['tier_match'] else '❌'}  "
          f"Action: {'✅' if results[-1]['action_match'] else '❌'}")

tier_acc   = sum(r["tier_match"]   for r in results) / len(results)
action_acc = sum(r["action_match"] for r in results) / len(results)
parse_rate = (len(results) - parse_failures) / len(results)

print(f"\n{'─'*40}")
print(f"  Risk Tier Accuracy : {tier_acc*100:.1f}%")
print(f"  Action Accuracy    : {action_acc*100:.1f}%")
print(f"  Pydantic Pass Rate : {parse_rate*100:.1f}%")
print(f"{'─'*40}")